In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

** <font size='6' color='red'>ch03. 연관분석</font> **
- pip install apyori
# 1. 연관분석 개요
- 데이터들 사이에 자주 발생하는 속성을 찾고, 그 속성들 사이에 연관성이 어느 정도 있는지 분석
- 활용분야 : 이벤트미리감지(사기적발..), 신상품카테고리

[<조건 : lerf-hand side> 오렌지 주스] -> [결과 :  right-hand side : 와인]
- 연관분석과 관련된 지표
1. 지지도(support) : 얼마나 자주 함께 나타나는지
    (lhs, rhs)의 항목수/전체항목수 = 0.2
2. 신뢰도(confidence) : 조건이 오면 결과가 얼마나 자주 나타나는지
    (lhs->rhs)의 항목수/lhs의 항목수 = 1/2 = 0.5
3. 향상도(lift) : 우연히 발생한 규칙은 아닌니 확인
    lhs -> rhs의 지지도 / (lhs의 지지도 *  rhs의 지지도) = 0.2 / (0.4*0.6) = 0.833
    향상도 < 1 : 기대가 낮다
    향상도 > 1 : 기대가 높다

# 2. 연관분석 구현

In [2]:
import csv
transation = []
with open('data/cf_basket.csv', 'r', encoding='utf-8') as f:
    csvdata = csv.reader(f)
    transation = list(csvdata)
transation

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [3]:
from apyori import apriori
rules = apriori(transation, # 2차원, 데이터
               min_support=0.15,
               min_confidence=0.1)
rules = list(rules)
len(rules)

18

In [4]:
rules[17]

RelationRecord(items=frozenset({'와인', '소주', '콜라'}), support=0.2, ordered_statistics=[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'와인', '소주', '콜라'}), confidence=0.2, lift=1.0), OrderedStatistic(items_base=frozenset({'소주'}), items_add=frozenset({'와인', '콜라'}), confidence=0.33333333333333337, lift=0.8333333333333334), OrderedStatistic(items_base=frozenset({'와인'}), items_add=frozenset({'소주', '콜라'}), confidence=0.33333333333333337, lift=0.5555555555555557), OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'와인', '소주'}), confidence=0.25, lift=1.25), OrderedStatistic(items_base=frozenset({'와인', '소주'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25), OrderedStatistic(items_base=frozenset({'소주', '콜라'}), items_add=frozenset({'와인'}), confidence=0.33333333333333337, lift=0.5555555555555557), OrderedStatistic(items_base=frozenset({'와인', '콜라'}), items_add=frozenset({'소주'}), confidence=0.5, lift=0.8333333333333334)])

In [5]:
rule = rules[10]
support = rule[1]
order_st = rule[2]
for item in order_st:
    lhs = item[0]
    rhs = item[1]
    confidence = item[2]
    lift = item[3]
    if lift > 1:
        print('{}=>{}\t {}\t {}\t {}'.format(lhs, rhs, support, 
                                             round(confidence,2),
                                             round(lift,2)))

frozenset({'소주'})=>frozenset({'콜라'})	 0.6	 1.0	 1.25
frozenset({'콜라'})=>frozenset({'소주'})	 0.6	 0.75	 1.25


In [6]:
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([data for data in item[0]])
        rhs = ', '.join([data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            print('{}=>{}\t {}\t {}\t {}'.format(lhs, rhs, support, 
                                                 round(confidence,2),
                                                 round(lift,2)))

맥주=>콜라	 0.4	 1.0	 1.25
콜라=>맥주	 0.4	 0.5	 1.25
소주=>콜라	 0.6	 1.0	 1.25
콜라=>소주	 0.6	 0.75	 1.25
콜라=>소주, 맥주	 0.2	 0.25	 1.25
소주, 맥주=>콜라	 0.2	 1.0	 1.25
맥주=>와인, 콜라	 0.2	 0.5	 1.25
콜라=>와인, 맥주	 0.2	 0.25	 1.25
와인, 맥주=>콜라	 0.2	 1.0	 1.25
와인, 콜라=>맥주	 0.2	 0.5	 1.25
소주=>오렌지주스, 콜라	 0.2	 0.33	 1.67
콜라=>소주, 오렌지주스	 0.2	 0.25	 1.25
소주, 오렌지주스=>콜라	 0.2	 1.0	 1.25
오렌지주스, 콜라=>소주	 0.2	 1.0	 1.67
콜라=>와인, 소주	 0.2	 0.25	 1.25
와인, 소주=>콜라	 0.2	 1.0	 1.25


In [7]:
import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs','rhs','지지도','신뢰도','향상도'])
# rules_df.loc[0] = ['와인','오렌지',0.15, 0.5, 1.1] 식으로 for문 내에서 데이터 추가
idx = 0
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([data for data in item[0]])
        rhs = ', '.join([data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            rules_df.loc[idx]=[lhs, rhs, support, round(confidence,2), round(lift,2)]
            idx += 1
rules_df.sort_values(by=['향상도','신뢰도'], ascending=False)

,lhs,rhs,지지도,신뢰도,향상도
13,"오렌지주스, 콜라",소주,0.2,1.00,1.67
10,소주,"오렌지주스, 콜라",0.2,0.33,1.67
0,맥주,콜라,0.4,1.00,1.25
2,소주,콜라,0.6,1.00,1.25
5,"소주, 맥주",콜라,0.2,1.00,1.25
8,"와인, 맥주",콜라,0.2,1.00,1.25
12,"소주, 오렌지주스",콜라,0.2,1.00,1.25
15,"와인, 소주",콜라,0.2,1.00,1.25
3,콜라,소주,0.6,0.75,1.25
1,콜라,맥주,0.4,0.50,1.25


# 3. 경주/전주 여행 자료 연관 분석

In [8]:
from konlpy.tag import Hannanum, Kkma, Komoran
queries = ['전주 여행', '경주 여행']
df = pd.DataFrame([])
for idx, query in enumerate(queries):
    df = pd.concat([df, pd.read_csv(f'data/naver_kin_{query}.csv', sep='\t')], ignore_index=True)
total_text_list = df['ttl_text'].to_list()
print(total_text_list[:2])
analyzer = Kkma()
total_noun_list = []
select_pos = ['NC', 'NQ'] # Hannanum 보통명사, 고유명사
select_pos = ['NNP', 'NNG'] # Kkma, Komoran 보통명사, 고유명사
불용어 = {'여행'}
for total_text in total_text_list:
#     total_noun = analyzer.nouns(total_text)
    total_nouns = [token for token, tag in analyzer.pos(total_text)
                     if tag in select_pos and token not in 불용어 and len(token)>1]
    total_noun_list.append(total_nouns)
print(total_noun_list[:2])

[' 경주여행 (숙소추천)  추억의 7080 다양한체험 7080감성 추억여행 테마박물관 유익한시간 2 전북 전북투어패스 통합이용권 전북핫플 여러여행지 다양한체험 카페이용추가 전주여행 필수 편안하고 즐거운 날이 되시길 바라겠습니다 감사합니다 ', ' 경주여행 (숙소추천)  전주여행 을 갈려고하는데요 아는사람과 갈려고하는데 호텔은 좋은가격에 정했고 음 2박3일여행인데 얼마정도갖고가면좋을까요 그리고 맛집같은거 카페같은거 추천해주세요 안녕하세요 전주 여행 계획 중이시네요 한옥마을 근처 ']
[['경주', '숙소', '추천', '추억', '다양', '체험', '추억', '테마', '박물관', '유익', '시간', '투어', '패스', '통합', '이용권', '여행지', '다양', '체험', '카페', '이용', '추가', '전주', '필수', '편안', '감사'], ['경주', '숙소', '추천', '전주', '사람', '호텔', '가격', '얼마', '정도', '카페', '추천', '주세', '안녕', '하세', '전주', '계획', '중이', '한옥', '마을', '근처']]


In [9]:
%%time
rules = apriori(total_noun_list, # 2차원, 데이터
               min_support=0.15,
               min_confidence=0.2)
rules = list(rules)
len(rules)

CPU times: total: 3.66 s
Wall time: 3.69 s


1591

In [ ]:
rules_df = pd.DataFrame(None, columns=['lhs','rhs','지지도','신뢰도','향상도'])
# rules_df.loc[0] = ['와인','오렌지',0.15, 0.5, 1.1] 식으로 for문 내에서 데이터 추가
idx = 0
for rule in rules:
    support = rule[1]
    order_st = rule[2]
    for item in order_st:
        lhs = ', '.join([data for data in item[0]])
        rhs = ', '.join([data for data in item[1]])
        confidence = item[2]
        lift = item[3]
        if lift > 1:
            rules_df.loc[idx]=[lhs, rhs, support, round(confidence,2), round(lift,2)]
            idx += 1
rules_df.sort_values(by=['향상도','신뢰도'], ascending=False)
rules_df = rules_df.reset_index(drop=True)

In [ ]:
pd.options.display.max_rows

In [ ]:
rules.df.head()